In [23]:
"""
نوتبوک تست جامع Pipeline
تاریخ: {امروز}
"""
import sys
from pathlib import Path

# اضافه کردن روت پروژه
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"📁 Project root: {PROJECT_ROOT}")

📁 Project root: c:\Users\programmer\Desktop\trade\src


In [24]:
from loguru import logger
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# تنظیمات نمایش
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)
pd.set_option('display.max_rows', 100)

from src.config import (
    load_config,
    load_symbols,
    get_target_symbols,
    get_symbol_sectors,
    get_db_path,
    setup_logging,
)
from src import db as dbm

setup_logging()
print("✅ Imports کامل")

✅ Imports کامل


In [25]:
import shutil
from datetime import datetime

db_file = get_db_path()
print(f"📍 DB: {db_file}")

if db_file.exists():
    backup_dir = db_file.parent / "backups"
    backup_dir.mkdir(exist_ok=True)
    
    dst = backup_dir / f"bourse_{datetime.now():%Y%m%d_%H%M%S}.db"
    shutil.copy2(db_file, dst)
    print(f"✅ بکاپ ساخته شد: {dst.name} ({dst.stat().st_size / 1024:.0f} KB)")
else:
    print("⚠️ دیتابیس هنوز ساخته نشده")

📍 DB: C:\Users\programmer\Desktop\trade\data\bourse.db
✅ بکاپ ساخته شد: bourse_20260914_170426.db (3444 KB)


In [26]:
# Config اصلی
cfg = load_config()
print(f"📋 Project: {cfg['project']['name']} v{cfg['project']['version']}")
print(f"📋 Markets: {cfg['markets']}")
print(f"📋 Data: {cfg['data']}")

print()

# Symbols
syms = load_symbols()
print(f"📋 Prediction: {syms['prediction']}")

targets = get_target_symbols()
sectors = get_symbol_sectors()
print(f"\n🎯 نمادهای هدف ({len(targets)}):")
for sector, s_list in syms['symbols'].items():
    print(f"   {sector}: {s_list}")

📋 Project: Bourse Predictor v0.1.0
📋 Markets: {'bourse': True, 'farabourse': False, 'payeh': False, 'haghe_taqadom': False, 'sandogh': False, 'bonds': False, 'options': False, 'mortgage': False, 'commodity': False, 'energy': False}
📋 Data: {'ingestion_interval_sec': 10, 'history_years': 3, 'rate_limit_delay_sec': 0.2, 'batch_size': 50}

📋 Prediction: {'horizon_days': 1, 'classes': ['up', 'down', 'neutral'], 'threshold_pct': 1.0}

🎯 نمادهای هدف (18):
   metals: ['فولاد', 'فخوز', 'ذوب']
   petrochemical: ['شپنا', 'شتران', 'تاپیکو']
   banking: ['وبملت', 'وتجارت', 'وبصادر']
   automotive: ['خودرو', 'خساپا']
   pharma: ['دارو', 'تیپیکو']
   telecom: ['همراه', 'اخابر']
   investment: ['وپارس', 'شستا']
   food: ['غبشهر']


In [27]:
from src.database import schema

# ⚠️ مهم: drop_existing=False چون نمی‌خوایم دیتا رو پاک کنیم
schema.create_database(drop_existing=False)

conn = dbm.connect()
tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
).fetchall()

print(f"📋 {len(tables)} جدول:")
for t in tables:
    name = t[0]
    if name == 'sqlite_sequence':
        continue
    count = dbm.table_count(conn, name)
    print(f"   {name:<25} {count:>10,} سطر")

conn.close()

17:04:26 | SUCCESS  | ✅ دیتابیس ساخته شد: data\bourse.db | 24 دستور اجرا شد


📋 12 جدول:
   client_type                       20 سطر
   codal_news                         0 سطر
   daily_history                     20 سطر
   index_companies                    0 سطر
   indices                            0 سطر
   instruments                      428 سطر
   intraday_candles                   0 سطر
   market_snapshot                    0 سطر
   order_book                         0 سطر
   queue                              0 سطر
   trades                             0 سطر


In [28]:
from src.data_ingestion.tsetmc_client import TsetmcClient

client = TsetmcClient(request_delay=0.2)

print("📡 get_symbols...")
symbols_df = client.get_symbols(bourse=True)
print(f"✅ {len(symbols_df)} ابزار")
print(f"\nستون‌ها: {symbols_df.columns.tolist()}")
print(f"\nنمونه:")
print(symbols_df.head(3).to_string())

📡 get_symbols...
Getting all of bourse symbols...
The dataframe of all bourse symbols is ready! (428 symbols)


17:04:42 | INFO     | تعداد 428 ابزار دریافت شد


✅ 428 ابزار

ستون‌ها: ['ins_code', 'symbol', 'name', 'english_name', 'company_code', 'company_isin', 'instrument_isin', 'asset_type', 'market', 'industry_group']

نمونه:
            ins_code symbol                           name    english_name company_code  company_isin instrument_isin asset_type                        market         industry_group
0  59612098290740355   آباد  توریستی ورفاهی آبادگران ایران       Abadgaran         ABAD  IRO1ABAD0002    IRO1ABAD0001      stock  بازار اول (تابلوی فرعی) بورس          هتل و رستوران
1  49054891736433700  دعبید   لابراتوارداروسازی  دکترعبیدی  Dr. Abidi Lab.         ABDI  IRO1ABDI0004    IRO1ABDI0001      stock  بازار اول (تابلوی فرعی) بورس  مواد و محصولات دارویی
2  70883594945615893  سآبیک                     سیمان آبیک    Abiak Cement         ABIK  IRO1ABIK0005    IRO1ABIK0001      stock  بازار اول (تابلوی اصلی) بورس        سیمان، آهک و گچ


In [29]:
print("📡 get_history (فولاد، ۱۵ روز)...")
hist = client.get_history(
    symbol="فولاد",
    start="2024-08-15",
    end="2024-09-15",
)
print(f"✅ {len(hist)} ردیف")
print(f"ستون‌ها: {hist.columns.tolist()}")
print(f"\n{hist.head(3).to_string()}")

📡 get_history (فولاد، ۱۵ روز)...
✅ 19 ردیف
ستون‌ها: ['gregorian_date', 'jalali_date', 'open', 'high', 'low', 'close', 'last', 'volume', 'value', 'trade_count']

   gregorian_date jalali_date  open  high   low  close  last     volume         value  trade_count
0        20240817  1403-05-27  4229  4229  4229   4229  4229   90397997  382293129313         1259
1        20240818  1403-05-28  4313  4313  4313   4313  4313   93710901  404175116013         1257
2        20240819  1403-05-29  4399  4399  4289   4300  4360  226562176  987717256069         5477


In [30]:
print("📡 get_client_type (فولاد، ۱۵ روز)...")
ct = client.get_client_type(
    symbol="فولاد",
    start="2024-08-15",
    end="2024-09-15",
)
print(f"✅ {len(ct)} ردیف")
print(f"ستون‌ها: {ct.columns.tolist()}")
print(f"\n{ct.head(2).to_string()}")

📡 get_client_type (فولاد، ۱۵ روز)...
✅ 19 ردیف
ستون‌ها: ['gregorian_date', 'jalali_date', 'n_buy_retail', 'n_buy_institutional', 'n_sell_retail', 'n_sell_institutional', 'vol_buy_retail', 'vol_buy_institutional', 'vol_sell_retail', 'vol_sell_institutional', 'val_buy_retail', 'val_buy_institutional', 'val_sell_retail', 'val_sell_institutional', 'per_capita_buy_retail', 'per_capita_sell_retail', 'per_capita_buy_institutional', 'per_capita_sell_institutional', 'power_retail', 'power_institutional']

   gregorian_date jalali_date  n_buy_retail  n_buy_institutional  n_sell_retail  n_sell_institutional  vol_buy_retail  vol_buy_institutional  vol_sell_retail  vol_sell_institutional  val_buy_retail  val_buy_institutional  val_sell_retail  val_sell_institutional  per_capita_buy_retail  per_capita_sell_retail  per_capita_buy_institutional  per_capita_sell_institutional  power_retail  power_institutional
0        20240817  1403-05-27           191                    2            582              

In [31]:
import algotik_tse as att

print("📡 get_market_snapshot...")
snap = att.get_market_snapshot()

print(f"\nنوع: {type(snap)}")
print(f"کلیدها: {list(snap.keys())}")
print(f"\nstocks shape: {snap['stocks'].shape}")
print(f"order_book shape: {snap['order_book'].shape}")
print(f"market_time: {snap['market_time']}")
print(f"index_value: {snap['index_value']}")
print(f"is_stale: {snap.get('is_stale')}")
print(f"age: {snap.get('snapshot_age_seconds'):.0f}s")

📡 get_market_snapshot...

نوع: <class 'dict'>
کلیدها: ['stocks', 'order_book', 'market_time', 'index_value', 'migration', 'trade_date', 'market_state', 'exchange_time', 'fetched_at', 'snapshot_age_seconds', 'is_today_trade_date', 'is_history_eligible', 'is_realtime_fresh', 'is_previous_trade_date', 'is_stale', 'is_partial']

stocks shape: (3626, 35)
order_book shape: (17067, 8)
market_time: 05/06/23 16:51:30
index_value: 7393247.13
is_stale: True
age: 805s


In [32]:
conn = dbm.connect()
daily_count = dbm.table_count(conn, "daily_history")
ct_count = dbm.table_count(conn, "client_type")
conn.close()

print(f"daily_history: {daily_count:,}")
print(f"client_type:   {ct_count:,}")

if daily_count < 1000:
    print("\n⚠️ دیتا کمه — Backfill لازمه")
    print("توی ترمینال بزن:")
    print("  python -m src.data_ingestion.backfill --years 3")
else:
    print("\n✅ دیتا کافیه")

daily_history: 20
client_type:   20

⚠️ دیتا کمه — Backfill لازمه
توی ترمینال بزن:
  python -m src.data_ingestion.backfill --years 3


In [33]:
# اجرای Collector در حالت force
import subprocess
result = subprocess.run(
    [sys.executable, "-m", "src.data_ingestion.realtime_fetcher",
     "--once", "--force", "--allow-stale"],
    capture_output=True, text=True, cwd=PROJECT_ROOT,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)


STDERR: c:\Users\programmer\anaconda3\python.exe: Error while finding module specification for 'src.data_ingestion.realtime_fetcher' (ModuleNotFoundError: No module named 'src')



In [34]:
conn = dbm.connect()
snap_count = dbm.table_count(conn, "market_snapshot")
book_count = dbm.table_count(conn, "order_book")
print(f"📸 market_snapshot: {snap_count}")
print(f"📖 order_book:      {book_count}")

if snap_count > 0:
    df = pd.read_sql("SELECT * FROM market_snapshot LIMIT 3", conn)
    print(f"\nنمونه:")
    print(df.to_string())
    
    # چک کردن نمادها
    symbols_in_snap = pd.read_sql("""
        SELECT DISTINCT i.symbol FROM market_snapshot m
        LEFT JOIN instruments i ON i.ins_code = m.ins_code
    """, conn)
    print(f"\nنمادهای ذخیره‌شده ({len(symbols_in_snap)}):")
    print(symbols_in_snap['symbol'].tolist())
    
conn.close()

📸 market_snapshot: 0
📖 order_book:      0


In [35]:
from src.preprocessing.cleaner import clean_history, detect_outliers
from src import db as dbm

# خواندن دیتای خام
raw_df = dbm.load_daily_history(symbols=["فولاد", "شپنا"])
print(f"خام: {len(raw_df)} ردیف")

# پاک‌سازی
clean_df = clean_history(raw_df, adjust=True, drop_outliers=True)
print(f"تمیز: {len(clean_df)} ردیف")

print(f"\nستون‌ها: {clean_df.columns.tolist()}")
print(f"\nنمونه:")
print(clean_df[['symbol', 'date', 'open', 'close', 'volume']].head(3).to_string())

17:04:55 | INFO     | clean_history: 20 → 20 سطر


خام: 20 ردیف
تمیز: 20 ردیف

ستون‌ها: ['symbol', 'ins_code', 'gregorian_date', 'jalali_date', 'open', 'high', 'low', 'close', 'last', 'volume', 'value', 'trade_count', 'date']

نمونه:
  symbol       date     open    close      volume
0   شپنا 2026-08-31  12600.0  12930.0  1381638913
1   شپنا 2026-09-01  13230.0  12510.0  1142818987
2   شپنا 2026-09-02  12520.0  12610.0  1136501326


In [36]:
outliers = detect_outliers(raw_df, method='iqr', iqr_multiplier=3.0)
print(f"تعداد outlierها: {outliers.sum()}")
if outliers.sum() > 0:
    print(f"\nنمونه outlierها:")
    print(raw_df[outliers].head().to_string())

تعداد outlierها: 0


In [37]:
from src.features.technical import compute_all_technical, feature_columns

tech_df = compute_all_technical(clean_df)
print(f"ستون‌های جدید: {len(tech_df.columns) - len(clean_df.columns)}")
print(f"\nستون‌های فیچر:")
new_cols = [c for c in tech_df.columns if c not in clean_df.columns]
for c in new_cols:
    print(f"   {c}")

print(f"\nنمونه:")
print(tech_df[['date', 'close', 'rsi', 'macd', 'bb_pct', 'atr_pct']].tail(3).to_string())

17:04:55 | INFO     | فیچرهای تکنیکال ساخته شد (29 ستون جدید)


ستون‌های جدید: 29

ستون‌های فیچر:
   sma_5
   close_to_sma_5
   sma_10
   close_to_sma_10
   sma_20
   close_to_sma_20
   sma_50
   close_to_sma_50
   ema_12
   ema_26
   rsi
   macd
   macd_signal
   macd_hist
   bb_mid
   bb_upper
   bb_lower
   bb_width
   bb_pct
   atr
   atr_pct
   stoch_k
   stoch_d
   return_1d
   return_5d
   return_20d
   volatility_20d
   volume_ratio_20d
   hl_range_pct

نمونه:
         date   close        rsi       macd   bb_pct   atr_pct
17 2026-09-09  2887.0  35.343701  10.183214      NaN  0.043385
18 2026-09-12  2967.0  42.836106  27.253673      NaN  0.041125
19 2026-09-13  3056.0  49.804884  47.417095  1.00478  0.039156


In [38]:
from src.features.seasonal import add_calendar_features, add_seasonal_decomposition

seasonal_df = add_calendar_features(tech_df)
print(f"ستون‌های تقویمی:")
calendar_cols = ['jalali_year', 'jalali_month', 'jalali_season', 'day_of_week',
                 'is_ramadan', 'is_muharram', 'month_sin', 'month_cos']
for c in calendar_cols:
    if c in seasonal_df.columns:
        print(f"   {c}: {seasonal_df[c].iloc[-1]}")

# تجزیه فصلی
seasonal_df = add_seasonal_decomposition(seasonal_df, column='close')
print(f"\nستون‌های تجزیه:")
for c in ['seasonal_trend', 'seasonal_seasonal', 'seasonal_resid']:
    if c in seasonal_df.columns:
        val = seasonal_df[c].iloc[-1]
        print(f"   {c}: {val:.2f}" if pd.notna(val) else f"   {c}: NaN")

17:04:55 | INFO     | فیچرهای تقویمی اضافه شد (20 سطر)
17:04:55 | WARNING  | داده برای تجزیه فصلی کافی نیست (10 < 44)
17:04:55 | WARNING  | داده برای تجزیه فصلی کافی نیست (10 < 44)


ستون‌های تقویمی:
   jalali_year: 1405
   jalali_month: 6
   jalali_season: 2
   day_of_week: 6
   is_ramadan: 0
   is_muharram: 0
   month_sin: 1.2246467991473532e-16
   month_cos: -1.0

ستون‌های تجزیه:
   seasonal_trend: NaN
   seasonal_seasonal: NaN
   seasonal_resid: NaN


In [39]:
from src.features.money_flow import compute_all_money_flow
from src import db as dbm

client_df = dbm.load_client_type(symbols=["فولاد", "شپنا"])
print(f"ورودی: {len(client_df)} ردیف")

mf_df = compute_all_money_flow(client_df)
print(f"خروجی: {len(mf_df)} ردیف، {len(mf_df.columns)} ستون")

mf_cols = [c for c in mf_df.columns if any(x in c for x in ['net_', 'imbalance', 'power_'])]
print(f"\nفیچرهای جریان پول ({len(mf_cols)}):")
for c in mf_cols[:10]:
    print(f"   {c}")

print(f"\nنمونه:")
print(mf_df[['symbol', 'date', 'net_val_retail', 'net_val_retail_ma5', 'power_ratio']].tail(3).to_string())

17:04:55 | INFO     | فیچرهای جریان پول ساخته شد (20 سطر)


ورودی: 20 ردیف
خروجی: 20 ردیف، 47 ستون

فیچرهای جریان پول (24):
   power_retail
   power_institutional
   net_val_retail
   net_val_institutional
   net_vol_retail
   net_vol_institutional
   net_val_retail_ratio
   net_val_inst_ratio
   net_val_retail_ma1
   net_val_inst_ma1

نمونه:
   symbol       date  net_val_retail  net_val_retail_ma5  power_ratio
17  فولاد 2026-09-09    3.626607e+11        5.922707e+11     1.747178
18  فولاد 2026-09-12    3.062679e+12        1.505446e+12   130.447761
19  فولاد 2026-09-13    5.404998e+12        2.508673e+12     4.510204


In [40]:
from src.features.order_book import compute_order_book_features

conn = dbm.connect()
book_df = pd.read_sql("SELECT * FROM order_book LIMIT 200", conn)
conn.close()

print(f"ورودی: {len(book_df)} ردیف")
print(f"ستون‌ها: {book_df.columns.tolist()}")

if not book_df.empty:
    ob_df = compute_order_book_features(book_df)
    print(f"\nخروجی: {len(ob_df)} ردیف")
    print(f"ستون‌ها: {ob_df.columns.tolist()}")
    print(f"\n{ob_df.head(3).to_string()}")

ورودی: 0 ردیف
ستون‌ها: ['id', 'ins_code', 'snapshot_time', 'level', 'bid_price', 'bid_volume', 'bid_order_count', 'ask_price', 'ask_volume', 'ask_order_count', 'created_at']


In [41]:
# ⚠️ توجه: این graph_builder فعلی، همبستگی می‌سازه نه جریان پول واقعی
# تصمیم: فعلاً skip می‌کنیم تا بعداً تصمیم بگیریم
from src.features.graph_builder import build_money_flow_graph

# فقط چک می‌کنیم import میشه
print("✅ graph_builder import شد")
print("⚠️ برای تست کامل، نیاز به تصمیم درباره معماری گراف داریم")

✅ graph_builder import شد
⚠️ برای تست کامل، نیاز به تصمیم درباره معماری گراف داریم


In [43]:
# Cell 21 (اصلاح‌شده): لیست توابع dataset.py
import src.training.dataset as ds
import inspect

print("📋 توابع و کلاس‌های dataset.py:")
print("=" * 60)

for name in sorted(dir(ds)):
    if name.startswith('_'):
        continue
    obj = getattr(ds, name)
    if inspect.isfunction(obj):
        sig = inspect.signature(obj)
        print(f"\n🔹 {name}{sig}")
        doc = inspect.getdoc(obj)
        if doc:
            first_line = doc.split('\n')[0]
            print(f"   {first_line}")
    elif inspect.isclass(obj):
        print(f"\n🏛️ class {name}")
        doc = inspect.getdoc(obj)
        if doc:
            first_line = doc.split('\n')[0]
            print(f"   {first_line}")

📋 توابع و کلاس‌های dataset.py:

🔹 add_labels(df: 'pd.DataFrame', horizon_days: 'Optional[int]' = None, threshold_pct: 'Optional[float]' = None) -> 'pd.DataFrame'
   افزودن ستون‌های ``future_return`` و ``target`` (۰ down، ۱ neutral، ۲ up).

🔹 build_sequences(df: 'pd.DataFrame', feature_cols: 'Optional[Sequence[str]]' = None, sequence_length: 'Optional[int]' = None) -> 'Dict[str, np.ndarray]'
   تبدیل جدول فیچر به دنباله‌های ``(نمونه, seq_len, n_features)``.

🔹 clean_client_type(df: 'pd.DataFrame', normalize: 'bool' = True) -> 'pd.DataFrame'
   پاک‌سازی داده حقیقی/حقوقی.

🔹 clean_history(df: 'pd.DataFrame', adjust: 'bool' = True, drop_outliers: 'bool' = True) -> 'pd.DataFrame'
   پاک‌سازی تاریخچه روزانه یک یا چند نماد.

🔹 compute_all_money_flow(client_df: 'pd.DataFrame') -> 'pd.DataFrame'
   اجرای زنجیره کامل فیچرهای جریان پول.

🔹 compute_all_seasonal(df: 'pd.DataFrame', with_sarima: 'bool' = False) -> 'pd.DataFrame'
   اجرای همه فیچرهای فصلی/تقویمی به‌صورت یکجا.

🔹 compute_all_technical

In [44]:
# چک کردن توابع موجود در dataset.py
import src.training.dataset as ds

funcs = [name for name in dir(ds) if not name.startswith('_') and callable(getattr(ds, name))]
print(f"توابع dataset.py:")
for f in funcs:
    print(f"   {f}")

توابع dataset.py:
   Dict
   List
   Optional
   Sequence
   Tuple
   add_labels
   build_sequences
   clean_client_type
   clean_history
   compute_all_money_flow
   compute_all_seasonal
   compute_all_technical
   feature_columns
   get_prediction_config
   get_section
   load_feature_table
   standardize
   walk_forward_splits


In [45]:
print("=" * 70)
print("📋 گزارش نهایی Pipeline")
print("=" * 70)

conn = dbm.connect()

print("\n📊 دیتابیس:")
for table in ["instruments", "daily_history", "client_type",
              "market_snapshot", "order_book", "codal_news"]:
    count = dbm.table_count(conn, table)
    status = "✅" if count > 0 else "⚠️"
    print(f"   {status} {table:<20}: {count:>10,}")

print("\n🎯 نمادهای هدف:")
targets = get_target_symbols()
print(f"   {len(targets)} نماد: {targets[:5]}...")

# چک کردن هر نماد توی instruments
print("\n🔍 وضعیت هر نماد:")
for sym in targets:
    row = conn.execute(
        "SELECT ins_code FROM instruments WHERE symbol = ?", (sym,)
    ).fetchone()
    if row:
        daily = conn.execute(
            "SELECT COUNT(*) FROM daily_history WHERE ins_code = ?", (row[0],)
        ).fetchone()[0]
        print(f"   ✅ {sym:<10} ins_code={row[0]:<20} daily={daily}")
    else:
        print(f"   ❌ {sym:<10} پیدا نشد")

conn.close()

print("\n" + "=" * 70)
print("✅ پایان تست")
print("=" * 70)

📋 گزارش نهایی Pipeline

📊 دیتابیس:
   ✅ instruments         :        428
   ✅ daily_history       :         20
   ✅ client_type         :         20
   ⚠️ market_snapshot     :          0
   ⚠️ order_book          :          0
   ⚠️ codal_news          :          0

🎯 نمادهای هدف:
   18 نماد: ['فولاد', 'فخوز', 'ذوب', 'شپنا', 'شتران']...

🔍 وضعیت هر نماد:
   ✅ فولاد      ins_code=46348559193224090    daily=10
   ✅ فخوز       ins_code=28864540805361867    daily=0
   ✅ ذوب        ins_code=71483646978964608    daily=0
   ✅ شپنا       ins_code=7745894403636165     daily=10
   ✅ شتران      ins_code=51617145873056483    daily=0
   ✅ تاپیکو     ins_code=22560050433388046    daily=0
   ✅ وبملت      ins_code=778253364357513      daily=0
   ✅ وتجارت     ins_code=63917421733088077    daily=0
   ✅ وبصادر     ins_code=28320293733348826    daily=0
   ✅ خودرو      ins_code=65883838195688438    daily=0
   ✅ خساپا      ins_code=44891482026867833    daily=0
   ✅ دارو       ins_code=67988012428906654    d

In [46]:
import plotly.graph_objects as go

df = dbm.load_daily_history(symbols=["فولاد"])
df = df.sort_values("date")

fig = go.Figure(data=[go.Candlestick(
    x=df["date"],
    open=df["open"], high=df["high"],
    low=df["low"], close=df["close"],
    name="فولاد"
)])
fig.update_layout(
    title="نمودار قیمت فولاد",
    xaxis_title="تاریخ",
    yaxis_title="قیمت (ریال)",
    height=500,
)
fig.show()